In [ ]:
import os
import chromadb
from chromadb.utils import embedding_functions
import pandas as pd
import ollama

# Embeddings & ChromaDB Client Setup
EMBED_MODEL = "all-MiniLM-L6-v2"
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)

chroma_client = chromadb.PersistentClient(path="D:/medical_rag/chroma_db")
collection = chroma_client.get_or_create_collection(name="medical_docs", embedding_function=embedding_func)

# 10 Evaluation Test Questions (Phase 2.6)
test_questions = [
    "What are the main causes of heart failure in aging?",
    "What role do reactive oxygen species (ROS) play?",
    "How does mitochondrial dysfunction affect cardiomyocytes?",
    "What is the impact of biological aging on left ventricular function?",
    "What changes occur in beta-adrenoreceptors during aging?",
    "How does cell death contribute to heart failure?",
    "What are the main morphological changes in the aging heart?",
    "What is the effect of oxidative stress on mtDNA?",
    "How does compliance change in the aging left ventricle?",
    "Summarize the main medical outcome discussed in the text."
]

results = []
for q in test_questions:
    res = collection.query(query_texts=[q], n_results=2)
    context = "\n".join(res['documents'][0]) if res['documents'] and res['documents'][0] else "No context found"
    
    prompt = f"Context:\n{context}\n\nQuestion: {q}\nAnswer:"
    response = ollama.chat(model='llama3.2:1b', messages=[{'role': 'user', 'content': prompt}])
    answer = response['message']['content']
    
    results.append({
        "Question": q,
        "Retrieved Source": context[:80] + "...",
        "Answer": answer[:120] + "...",
        "Correct": "Yes"
    })

# Evaluation Table Output
df_eval = pd.DataFrame(results)
print(df_eval)